# 対話の履歴を管理して、長くなったら古いターンを捨てる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_dialogue/chat_history.ipynb)

NLP クックブック 25-05 ／ 対話システム

LLM と何ターンも対話を続けたい。履歴が長くなると入力が膨らんで遅くなり、やがて上限を超えるので、長さを管理したい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。CPU でも動く（メモリ約 7GB）
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

class Chat:
    def __init__(self, system, max_input_tokens=300):
        self.system = {"role": "system", "content": system}
        self.history = []                       # user と assistant の発話を交互に積む
        self.max_input_tokens = max_input_tokens

    def n_tokens(self, messages):
        return len(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True)["input_ids"])

    def window(self):
        """システムプロンプトは必ず残し、上限に収まるまで古いターン（user と assistant の組）を捨てる"""
        history = list(self.history)
        while len(history) > 1 and self.n_tokens([self.system] + history) > self.max_input_tokens:
            history = history[2:]
        return [self.system] + history

    def say(self, text, max_new_tokens=150):
        self.history.append({"role": "user", "content": text})
        messages = self.window()
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        self.history.append({"role": "assistant", "content": reply})
        return reply, inputs["input_ids"].shape[1], (len(messages) - 1) // 2

chat = Chat("あなたは旅行の相談に乗るアシスタントです。1〜2 文で短く答えてください。")
for user in ["私の名前は田中です。来月、京都に 2 泊で行く予定です。", "秋におすすめのお寺を 1 つ教えてください。", "そこへは京都駅からどう行けばいいですか？",
             "近くでお昼を食べるなら何がいいですか？", "最後に確認です。私の名前は何でしたか？"]:
    reply, n_in, n_turns = chat.say(user)
    print(f"あなた: {user}\nボット: {reply}\n   (入力 {n_in} トークン、直近 {n_turns} ターン分を使用)")

あなた: 私の名前は田中です。来月、京都に 2 泊で行く予定です。
ボット: 田中様、京都へのご旅行ですね！来月であれば、桜の季節や秋の紅葉シーズンは過ぎていますが、それでも美しい景色や様々な観光スポットを楽しむことができます。おすすめのプランやスポットがありましたら、ぜひ教えてください。
   (入力 38 トークン、直近 0 ターン分を使用)
あなた: 秋におすすめのお寺を 1 つ教えてください。
ボット: 秋に京都を訪れるなら、嵐山にある「宝厳院」がおすすめです。紅葉の時期には多くのもみじが境内を彩り、stunningな風景が広がります。また、獅子吼の庭と呼ばれる庭園は、ライトアップも行われ、幻想的な雰囲気を楽しむことができます。
   (入力 92 トークン、直近 1 ターン分を使用)
あなた: そこへは京都駅からどう行けばいいですか？
ボット: 宝厳院へは、京都駅からJR嵯峨野線（山陰本線）を利用してお越しいただくのが便利です。京都駅から「嵯峨嵐山駅」まで約30分で到着します。嵯峨嵐山駅からは、徒歩約20分ほどで宝厳院に到着します。もしバスを利用される場合は、京都駅から市バス28系統または72系統に乗車し、「嵐山天龍寺前」バス停で下車、徒歩約5分です。
   (入力 152 トークン、直近 2 ターン分を使用)
あなた: 近くでお昼を食べるなら何がいいですか？
ボット: 嵐山エリアでは、京野菜や湯葉を使った料理が人気です。お昼を食べるなら、嵐山渡月橋近くの「湯葉料理専門店 奥丹」や「嵐山辨慶」などがおすすめです。どちらも湯葉料理を中心に、京の味覚を堪能することができます。また、竹林の小径 nearbyの「阿じろ」では、京弁当や川魚料理を楽しむことができます。いずれも人気店ですので、事前に予約されることをおすすめします。
   (入力 261 トークン、直近 3 ターン分を使用)
あなた: 最後に確認です。私の名前は何でしたか？
ボット: 申し訳ありませんが、会話の中でお名前について触れられていなかったため、お名前が分かりません。もし差し支えなければ、今後のためにお名前を教えていただけますか？それとも、別の質問があればお答えすることができます。
   (入力 298 トークン、直近 3 ターン分を使用)


- LLM は状態を持たない。「覚えている」ように見えるのは、毎回これまでの対話をすべて入力しているから。
- 古いターンを捨てると、そこで話したこと（最初に名乗った名前）は忘れる。上限を小さくしてあるこの例では、最後の質問でそれが起きる。
- 「1〜2 文で」と指示しても、このモデルは長めに答えることが多い。`max_new_tokens` は生成するトークン数の上限で、そこに達すると文の途中で切れる。ここでは応答が途中で切れないように 150 にしてある。応答が長いほど履歴もすぐ長くなる。
- 店名やバスの系統などの細かい事実は、モデルが知識から作ったもので、誤りを含むことがある。
- 捨てるときは user と assistant の組で捨てる。片方だけ残すと、チャットテンプレートの交互の並びが崩れる。
- 大事な情報を失わないための方法が、次項の「古い対話を要約して長い対話を続ける」。
- トークナイザーを読み込んだあとの数行は、transformers 5 系で sarashina2.2 の分割が学習時と変わってしまう不具合への対処（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照）。この章のほかのレシピも同じ。